# Project 4: Scrapping Data from Indeed.com 

##### I build two key functions in this notebook

*DataScienceJobSearch*
* This function crawls across Indeed.com scrapping key data points from data scientist job posts in cities of your choice

*SQLFunc*
* This function was built as an exercise in streaming data to a local psql database

In [20]:
import requests
import urllib2
import bs4
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import psycopg2
import string

In [12]:
df = pd.DataFrame()
pd.set_option('max_colwidth', 500)
df

""


In [13]:
#saving this function for another day

def SQLFunc(x):

    conn = psycopg2.connect(host = "localhost",
                            database="allenbyron",
                            port=5432)

    cur = conn.cursor()

    #Create sql query and execute
    sql_insert = 'insert into data_science_jobs (' + string.join(x[0], ', ') + ') values(' + ('%s, ' * (len(x) - 1)) + '%s' + ');'
    cur.execute(sql_insert, [i for i in x[1]])

    conn.commit()
    cur.close()
    conn.close()

In [14]:
def DataScienceJobSearch(city,df):
    
    jobs_scrapped = 0 #not useful when salary bins are taken into account 

    for c in city:        
        if c == 'Sydney' or c == 'Melbourne':
            home_url = 'http://au.indeed.com'
            base_url = 'http://au.indeed.com/jobs?q=data+scientist'
            loc_url = base_url + '&l=' + c
        else:
            home_url = 'http://www.indeed.com'
            base_url = 'http://www.indeed.com/jobs?q=data+scientist'
            loc_url = base_url + '&l=' + c
        
        #Identify salary bins
        estimated_salary = BeautifulSoup(urllib2.urlopen(loc_url), "lxml")
        rbList = estimated_salary.find('ul', attrs={'class': 'rbList'})
        salary_tuple = zip([i.getText() for i in rbList.find_all('a')],[i.get('href') for i in rbList.find_all('a')]) 
        print loc_url
        
        for s in salary_tuple:
            
            #Define salary bins
            salary_bins = s[0]
            bin_as_url_elem = salary_bins.split(',')[0] 
            
            salary_url = home_url + s[1]
            print salary_url
            content_for_range = BeautifulSoup(urllib2.urlopen(salary_url), "lxml")

            #Define max of range
            try:
                searchCount = content_for_range.find('div', attrs={'id': 'searchCount'}).getText().strip()
                number_of_results = str(searchCount).split(' ')
                max_of_range = int(number_of_results[-1]) 
                var = int(round((max_of_range/10)) + 2)
            except:
                var = 100
            
            #change max of range to var if not var
            for page in range(1, var): 
                page = (page-1) * 10  
                start_of_url = '/jobs?q=data+scientist+%24'
                mid_url = '%2C000&l='
                start_from = '&start='  # start page number
                url = "%s%s%s%s%s%s%d" % (home_url, start_of_url, bin_as_url_elem, mid_url, c, start_from, page) 
                print url
                
                contents = BeautifulSoup(urllib2.urlopen(url), "lxml") 
                contentsElements = contents.find_all('div', attrs={'class': ' row result'}) 

                for job in contentsElements:
                    
                    jobs_scrapped += 1         
                    
                    #Details of the job
                    try: job_title = job.find('h2', attrs={'class':'jobtitle'}).getText().strip()
                    except: job_title = '-' #None

                    try: company = job.find('span', attrs={'itemprop':'name'}).getText().strip()
                    except: company = '-' #None

                    try: salary = job.find('nobr').getText().strip() 
                    except: salary = '-' #None

                    try: reviews_total = job.find('span', attrs={'class':'slNoUnderline'}).getText().strip()
                    except: reviews_total = '-' #None

                    try: job_description = job.find('span', attrs={'itemprop':'description'}).getText()
                    except: job_description = '-' #None

                    try: link = "%s%s" % (home_url, job.find('a').get('href'))
                    except: link = '-' #None

                    try: job_addr = job.find('span', attrs={'itemprop':'addressLocality'}).getText()
                    except: job_addr = '-' #None

                    try: job_posted = job.find('span', attrs={'class': 'date'}).getText()   
                    except: job_posted = '-' #None



                    #Company page
                    try:
                        indeed_company_url = "%s%s%s" % (home_url, '/cmp/', company.replace(' ','-'))
                        indeed_company_page = BeautifulSoup(urllib2.urlopen(indeed_company_url), "lxml")
                    except:
                        #print Exception, 'problem with indeed_company_url', company
                        continue


                    #Glean overall company rating 
                    try:
                        company_overall_ratingElements = indeed_company_page.find_all('div', attrs={'id':'cmp-reviews'})
                        for score in company_overall_ratingElements:
                            reviews_score = score.find('span', attrs={'class':'cmp-average-rating'}).getText().strip()
                    except:
                        reviews_score = '-' #None


                    #Glean various company ratings
                    try:
                        specific_ratingsElements = indeed_company_page.find('dl', attrs={'id':'cmp-reviews-attributes'})    
                        specific_ratingsElements = specific_ratingsElements.find_all('span', attrs={'class': 'cmp-star-rating'})

                        if specific_ratingsElements != None:

                            #print 'specific_ratingsElements'

                            if specific_ratingsElements[0] != None:
                                #print 'first element has something'

                                try: work_life_bal = specific_ratingsElements[0].getText()
                                except: work_life_bal = '-' #None

                                try: salary_benefits = specific_ratingsElements[1].getText()
                                except: salary_benefits = '-' #None

                                try: job_security_advance = specific_ratingsElements[2].getText()
                                except: job_security_advance = '-' #None

                                try: management = specific_ratingsElements[3].getText()
                                except: management = '-' #None

                                try: culture = specific_ratingsElements[4].getText()
                                except: culture = '-' #None        

                            else:
                                #print 'first element has nothing'

                                try: work_life_bal = specific_ratingsElements[1].getText()
                                except: work_life_bal = '-' #None

                                try: salary_benefits = specific_ratingsElements[2].getText()
                                except: salary_benefits = '-' #None

                                try: job_security_advance = specific_ratingsElements[3].getText()
                                except: job_security_advance = '-' #None

                                try: management = specific_ratingsElements[4].getText()
                                except: management = '-' #None

                                try: culture = specific_ratingsElements[5].getText()
                                except: culture = '-' #None   

                    except:
                        #print 'specific_ratingsElements is empty'

                        work_life_bal = '-' #None
                        salary_benefits = '-' #None
                        job_security_advance = '-' #None
                        management = '-' #None
                        culture = '-' #None   

                    #print '...........................'

                    df = df.append({'job_title': job_title,
                                    'company': company,
                                    'salary': salary,
                                    'reviews_total': reviews_total,
                                    'reviews_score': reviews_score,
                                    'job_description': job_description,
                                    'link': link,
                                    'job_addr': job_addr,
                                    'job_posted': job_posted,
                                    'work_life_bal': work_life_bal,
                                    'salary_benefits': salary_benefits,
                                    'job_security_advance': job_security_advance,
                                    'management': management,
                                    'culture': culture,
                                    'salary_bins': salary_bins
                                    }, ignore_index=True)
                    
                    #SQLFunc #Alternative choice to df.append()
    
    return df

In [15]:
list_of_cities = ['Austin'] #'Sydney','Melbourne', 'San+Francisco', 'Austin', 'Chicago'
job_searches = DataScienceJobSearch(list_of_cities,df)
job_searches

http://www.indeed.com/jobs?q=data+scientist&l=Chicago
http://www.indeed.com/q-data-scientist-$55,000-l-Chicago-jobs.html
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=0
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=10
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=20
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=30
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=40
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=50
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=60
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=70
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=80
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=90
http://www.indeed.com/jobs?q=data+scientist+%24$55%2C000&l=Chicago&start=100
http://www.indeed.com/jobs?q=data+scientist

,company,culture,job_addr,job_description,job_posted,job_security_advance,job_title,link,management,reviews_score,reviews_total,salary,salary_benefits,salary_bins,work_life_bal
0,The University of Chicago Medicine,3.8,"Chicago, IL","\nThe Data Scientist will also be responsible for data profiling/cleansing data governance consultation, data quality assessment, data mart mapping and data...",13 hours ago,3.8,Data Scientist,http://www.indeed.com/rc/clk?jk=6a38a0586190a6d7&fccid=46f88c38f8b3d84f,3.6,4.3,7 reviews,-,4.0,"$55,000+",4.0
1,Amyx,-,"Chicago, IL","\nAmyx is seeking to hire a Data Scientist in Chicago, IL to support a large contract. Seeking a Senior Data Scientist with experience and knowledge of analytics...",3 days ago,-,Data Scientist,http://www.indeed.com/rc/clk?jk=3e0070b7e8dc8931&fccid=c8065917ab1f5e6e,-,4.3,-,-,-,"$55,000+",-
2,RAY ALLEN,-,"Chicago, IL 60654 (Loop area)","\nIn need of a detail-oriented, analytical problem solver for a full-time Associate Data Scientist. 2+ years Data Analysis experience....",2 days ago,-,Associate Data Scientist,http://www.indeed.com/rc/clk?jk=3a982c164ab623ef&fccid=e5a2c06c5d17f0c6,-,4.3,-,-,-,"$55,000+",-
3,ZirMed,4.0,"Chicago, IL 60606 (Loop area)","\nWe are looking for an experienced Data Scientist, who has previously supported healthcare software applications....",8 days ago,3.5,Data Scientist,http://www.indeed.com/rc/clk?jk=28c603e910a47aed&fccid=7e1b7be66e59de2d,4.3,4.0,5 reviews,-,4.5,"$55,000+",4.3
4,Valence Health,2.9,"Chicago, IL",\nWe are looking for bright and energetic individuals to fill a data scientist position in our Analytic Services department....,19 days ago,2.3,Data Scientist,http://www.indeed.com/rc/clk?jk=31a10e2ccf371306&fccid=295ae525d65b3dc9,2.5,2.9,-,-,3.0,"$55,000+",2.8
5,Caterpillar,3.7,"Chicago, IL 60606 (Loop area)",\nCaterpillar is hiring the best data scientists to join its newly opened Digital & Analytics Hub within the Merchandise Mart in Chicago....,13 days ago,3.2,Data Scientist,http://www.indeed.com/rc/clk?jk=1949ed11529366f6&fccid=d723225da214c842,3.4,3.9,"1,964 reviews",-,3.9,"$55,000+",3.8
6,Cognizant,3.9,"Chicago, IL",\nScale data manipulation and mining/pattern recognition. Statistical analysis large scale data manipulation Monte Carlo simulation. Job....,16 days ago,4.0,Data Scientist - Economics,http://www.indeed.com/rc/clk?jk=6900ecfb39fa4892&fccid=2df6a1e69a70a1e7,3.6,4.0,"4,678 reviews",-,3.5,"$55,000+",3.8
7,Wolverine Trading,5.0,"Chicago, IL 60604 (Loop area)","\nMachine Learning Data Scientist. You will be using large data sets to create trading strategies, improve trading, quantitative research and data mining....",15 days ago,3.0,Machine Learning Data Scientist,http://www.indeed.com/rc/clk?jk=4b77ac5b55eefbce&fccid=1ec8c83c883d1273,4.0,4.0,-,-,3.0,"$55,000+",5.0
8,Dupage Medical Group,2.9,"Downers Grove, IL",\n3+ years' experience analyzing large data sets. Experience with healthcare data – both clinical and financial....,8 hours ago,2.6,Data Scientist-Analytics,http://www.indeed.com/rc/clk?jk=1b54efb6b0ab5ee2&fccid=7c91c4b781cb8588,2.6,3.1,70 reviews,-,2.7,"$55,000+",3.2
9,Dupage Medical Group,2.9,"Downers Grove, IL",\n3+ years' experience analyzing large data sets. Experience with healthcare data – both clinical and financial....,8 hours ago,2.6,Data Scientist-Analytics,http://www.indeed.com/rc/clk?jk=1b54efb6b0ab5ee2&fccid=7c91c4b781cb8588,2.6,3.1,70 reviews,-,2.7,"$55,000+",3.2


In [16]:
print job_searches.shape
job_searches.drop_duplicates(subset = ['company', 'job_description', 'job_title', 'link'], keep='last',inplace=True)

(1044, 15)


In [17]:
job_searches.reviews_score = job_searches.reviews_score.astype(np.float_)
job_searches['salary_bins_int'] = [x.replace('$', '').replace(',', '').replace('+','') for x in job_searches.salary_bins]
job_searches.reviews_total = [x.replace(' reviews', '') for x in job_searches.reviews_total]
job_searches.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 309 entries, 2 to 1043
Data columns (total 16 columns):
company                 309 non-null object
culture                 309 non-null object
job_addr                309 non-null object
job_description         309 non-null object
job_posted              309 non-null object
job_security_advance    309 non-null object
job_title               309 non-null object
link                    309 non-null object
management              309 non-null object
reviews_score           309 non-null float64
reviews_total           309 non-null object
salary                  309 non-null object
salary_benefits         309 non-null object
salary_bins             309 non-null object
work_life_bal           309 non-null object
salary_bins_int         309 non-null object
dtypes: float64(1), object(15)
memory usage: 41.0+ KB


In [18]:
import sys
reload(sys)
sys.setdefaultencoding('utf-8')
job_searches.to_csv('chi_job_searches.csv', index=False)